# Paper 2 — IEEE Conference Paper
# Comparative Analysis of Machine Learning Algorithms for Crop Recommendation Systems

**Complete reproducible pipeline. All figures generated at 600 DPI in both PNG and PDF.**

This notebook produces every figure and table needed for an 8-page IEEE conference paper:

| # | Figure | What it shows | Paper section |
|---|--------|----------------|---------------|
| 1 | Methodology Flowchart | Block diagram of the pipeline | III. Methodology |
| 2 | Class Distribution | Sample count per crop | III. Dataset |
| 3 | Feature Distributions | Violin plots of 7 features | III. Dataset |
| 4 | Correlation Heatmap | Pairwise feature correlations | III. Dataset |
| 5 | PCA 2D Projection | Class separability in 2D | III. Dataset |
| 6 | Cross-Validation Boxplot | 5-fold CV variance per model | IV. Results |
| 7 | Performance Bar Chart | Accuracy / Precision / Recall / F1 | IV. Results |
| 8 | Training Time | Time-vs-accuracy tradeoff | IV. Results |
| 9 | Confusion Matrix | Best model, 22×22 | IV. Results |
| 10 | Feature Importance | Driver features | IV. Results |
| 11 | ROC Curves | Macro & per-class (one-vs-rest) | IV. Results |
| 12 | Learning Curve | Train vs validation accuracy | IV. Results |
| 13 | Ensemble Architecture | Proposed system | V. Proposed Approach |

**Dataset:** Crop Recommendation Dataset (Kaggle, 2200 rows × 8 cols, 22 crop classes).
Search "Crop Recommendation Dataset Atharva Ingle" on Kaggle, or find a GitHub mirror.

**Total runtime:** ~3 minutes on Colab free CPU. No GPU needed.

## How to run

1. Download **`Crop_recommendation.csv`** from Kaggle and place it in the same folder as this notebook: https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset
2. Install dependencies: `pip install -r requirements.txt`
3. Run all cells top to bottom.

**Note:** the bagging and 10-seed robustness experiments each train models over ten random splits, so those two cells take a few minutes on a CPU. All splits use a fixed seed (42) for reproducibility.

## 0. Setup

### 0.1 Install dependencies

In [ ]:
# xgboost may need installing; the rest are standard scientific-Python packages
!pip install xgboost -q

### 0.2 Imports & IEEE plot style

In [ ]:
import os
import time
import warnings
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, learning_curve, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, roc_curve, auc)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
np.random.seed(42)

# IEEE-style matplotlib defaults (sans-serif, proper font embedding, 600 DPI output)
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 110,        # display
    'savefig.dpi': 600,       # save
    'pdf.fonttype': 42,       # embed fonts as TrueType (IEEE PDF requirement)
    'ps.fonttype': 42,
    'axes.linewidth': 0.8,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'grid.linewidth': 0.5,
})

print('Libraries loaded. Matplotlib version:', plt.matplotlib.__version__)

### 0.3 Output directory & helper to save every figure as PNG + PDF at 600 DPI

In [ ]:
FIG_DIR = 'figures'
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(fig, name, dpi=600):
    """Save figure as both 600-DPI PNG and vector PDF (for IEEE submission)."""
    png_path = f'{FIG_DIR}/{name}.png'
    pdf_path = f'{FIG_DIR}/{name}.pdf'
    fig.savefig(png_path, dpi=dpi, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
    print(f'  saved: {png_path} + {pdf_path}')

print(f'Figures will be saved to: ./{FIG_DIR}/')

## 1. Methodology — Figure 1

Block diagram of the complete pipeline. Always include this in IEEE methodology section.

In [ ]:
def draw_box(ax, xy, w, h, text, facecolor, fontsize=8, fontweight='normal'):
    box = FancyBboxPatch(xy, w, h, boxstyle='round,pad=0.05,rounding_size=0.15',
                         linewidth=1.2, edgecolor='black', facecolor=facecolor)
    ax.add_patch(box)
    ax.text(xy[0] + w/2, xy[1] + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight=fontweight, wrap=True)

def draw_arrow(ax, start, end, color='black'):
    arrow = FancyArrowPatch(start, end,
                            arrowstyle='->,head_length=8,head_width=6',
                            linewidth=1.5, color=color)
    ax.add_patch(arrow)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.set_xlim(0, 12); ax.set_ylim(0, 4.5); ax.axis('off')

box_w, box_h = 1.9, 1.6
boxes = [
    {'xy': (0.1, 1.5), 'text': 'Crop Recommendation\nDataset\n(2200 samples,\n7 features,\n22 classes)', 'color': '#AED6F1'},
    {'xy': (2.3, 1.5), 'text': 'Data\nPreprocessing\n(Label Encoding,\nStandardization)', 'color': '#A9DFBF'},
    {'xy': (4.5, 1.5), 'text': 'Train/Test\nSplit\n(80% / 20%,\nstratified)', 'color': '#F9E79F'},
    {'xy': (6.7, 1.5), 'text': '8 ML Classifiers\nTrained\n(LR, DT, RF, SVM,\nKNN, NB, GB, XGB)', 'color': '#F5B7B1'},
    {'xy': (8.9, 1.5), 'text': 'Evaluation\n(Accuracy, Precision,\nRecall, F1-score,\nConfusion Matrix)', 'color': '#D2B4DE'},
]
for b in boxes:
    draw_box(ax, b['xy'], box_w, box_h, b['text'], b['color'])
for i in range(len(boxes) - 1):
    sx = boxes[i]['xy'][0] + box_w
    sy = boxes[i]['xy'][1] + box_h / 2
    draw_arrow(ax, (sx + 0.02, sy), (boxes[i+1]['xy'][0] - 0.02, sy))

draw_box(ax, (8.9, 0.2), 1.9, 0.8, 'Best Model\nSelection', '#FAD7A0',
         fontsize=8, fontweight='bold')
draw_arrow(ax, (9.85, 1.5), (9.85, 1.0))

plt.title('Proposed Methodology', fontsize=11, fontweight='bold', pad=10)
plt.tight_layout()
save_fig(fig, 'fig01_methodology')
plt.show()

## 2. Dataset Loading & Exploratory Analysis

### 2.1 Upload the CSV

When prompted, upload `Crop_recommendation.csv`.

In [ ]:
# Download Crop_recommendation.csv from Kaggle and place it next to this notebook:
# https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset
df = pd.read_csv('Crop_recommendation.csv')
print('Shape:', df.shape)
df.head()

### 2.2 Basic statistics for the paper's Dataset section

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing values ===')
print(df.isnull().sum())
print('\n=== Statistical summary ===')
display(df.describe().round(2))
print(f'\nNumber of classes (crops): {df["label"].nunique()}')
print(f'Total samples: {len(df)}')
print(f'Samples per class: {df["label"].value_counts().iloc[0]} (balanced)')

### Figure 2 — Class distribution

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
counts = df['label'].value_counts().sort_values(ascending=False)
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(counts)))
bars = ax.bar(range(len(counts)), counts.values, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(counts)))
ax.set_xticklabels(counts.index, rotation=45, ha='right')
ax.set_ylabel('Number of samples')
ax.set_xlabel('Crop type')
ax.set_title('Class Distribution across 22 Crop Types', fontweight='bold')
ax.set_ylim(0, max(counts.values) * 1.15)
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 1, str(v),
            ha='center', va='bottom', fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
save_fig(fig, 'fig02_class_distribution')
plt.show()

### Figure 3 — Feature distributions (violin plots)

In [ ]:
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
axes = axes.flatten()
for i, feat in enumerate(features):
    parts = axes[i].violinplot(df[feat].values, showmeans=True, showmedians=True)
    for pc in parts['bodies']:
        pc.set_facecolor('#5DADE2')
        pc.set_edgecolor('black')
        pc.set_alpha(0.7)
    axes[i].set_title(feat, fontweight='bold')
    axes[i].set_ylabel('Value')
    axes[i].set_xticks([])
    axes[i].grid(alpha=0.3)
axes[7].axis('off')   # hide 8th subplot
plt.suptitle('Distribution of Soil & Climate Features', fontsize=12, fontweight='bold', y=1.0)
plt.tight_layout()
save_fig(fig, 'fig03_feature_distributions')
plt.show()

### Figure 4 — Feature correlation heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
corr = df.drop('label', axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f',
            square=True, linewidths=0.5, linecolor='white',
            cbar_kws={'shrink': 0.75, 'label': 'Pearson r'},
            vmin=-1, vmax=1, ax=ax, annot_kws={'fontsize': 9})
ax.set_title('Feature Correlation Heatmap', fontweight='bold', pad=10)
plt.tight_layout()
save_fig(fig, 'fig04_correlation_heatmap')
plt.show()

### Figure 5 — PCA 2D projection showing class separability

In [ ]:
X_raw = df.drop('label', axis=1).values
y_raw = df['label'].values
scaler_pca = StandardScaler()
X_scaled_pca = scaler_pca.fit_transform(X_raw)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled_pca)

fig, ax = plt.subplots(figsize=(9, 6.5))
unique_crops = sorted(np.unique(y_raw))
palette = sns.color_palette('tab20', n_colors=len(unique_crops))
for i, crop in enumerate(unique_crops):
    mask = y_raw == crop
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               s=22, alpha=0.75, color=palette[i], label=crop,
               edgecolor='white', linewidth=0.3)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title('PCA 2D Projection of Crop Recommendation Dataset', fontweight='bold')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5),
          ncol=1, fontsize=8, frameon=True)
ax.grid(alpha=0.3)
plt.tight_layout()
save_fig(fig, 'fig05_pca_projection')
plt.show()
print(f'\nTotal variance explained by first 2 PCs: {sum(pca.explained_variance_ratio_)*100:.1f}%')

## 3. Preprocessing & Train/Test Split

In [ ]:
X = df.drop('label', axis=1)
y = df['label']

# Encode crop labels to integers
le = LabelEncoder()
y_enc = le.fit_transform(y)
class_names = le.classes_

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# Standardize features (needed for LR, SVM, KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training set: {X_train.shape}, Test set: {X_test.shape}')
print(f'Number of classes: {len(class_names)}')

## 4. Train 8 Classical ML Models

Each model trained with default hyperparameters (paper note: 'we use default hyperparameters from scikit-learn' is a standard, acceptable choice for a comparative study).

In [ ]:
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), True),
    'Decision Tree':       (DecisionTreeClassifier(random_state=42), False),
    'Random Forest':       (RandomForestClassifier(n_estimators=100, random_state=42), False),
    'SVM (RBF)':           (SVC(kernel='rbf', probability=True, random_state=42), True),
    'KNN':                 (KNeighborsClassifier(n_neighbors=5), True),
    'Naive Bayes':         (GaussianNB(), False),
    'Gradient Boosting':   (GradientBoostingClassifier(random_state=42), False),
    'XGBoost':             (XGBClassifier(eval_metric='mlogloss', random_state=42,
                                          ), False),
}

results = []
trained = {}
for name, (model, needs_scaling) in models.items():
    Xtr, Xte = (X_train_scaled, X_test_scaled) if needs_scaling else (X_train.values, X_test.values)
    t0 = time.time()
    model.fit(Xtr, y_train)
    train_time = time.time() - t0
    y_pred = model.predict(Xte)
    results.append({
        'Model':      name,
        'Accuracy':   accuracy_score(y_test, y_pred),
        'Precision':  precision_score(y_test, y_pred, average='weighted'),
        'Recall':     recall_score(y_test, y_pred, average='weighted'),
        'F1-score':   f1_score(y_test, y_pred, average='weighted'),
        'Train_time': train_time,
    })
    trained[name] = (model, needs_scaling)
    print(f'  {name:25s}  acc={results[-1]["Accuracy"]:.4f}  t={train_time:.3f}s')

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False).reset_index(drop=True)
print('\n=== TABLE I: Model Performance Comparison ===\n')
display(results_df.round(4))
results_df.round(4).to_csv(f'{FIG_DIR}/table1_model_comparison.csv', index=False)

### Figure 6 — 5-Fold Cross-Validation Comparison

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = {}
print('Running 5-fold CV for each model...')
for name, (model, needs_scaling) in models.items():
    Xtr = X_train_scaled if needs_scaling else X_train.values
    scores = cross_val_score(model, Xtr, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_scores[name] = scores
    print(f'  {name:25s}  {scores.mean():.4f} +/- {scores.std():.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ordered = results_df['Model'].tolist()
data = [cv_scores[m] for m in ordered]
bp = ax.boxplot(data, labels=ordered, patch_artist=True, widths=0.6, showmeans=True,
                meanprops={'marker':'D', 'markerfacecolor':'red', 'markeredgecolor':'red'})
palette = plt.cm.viridis(np.linspace(0.2, 0.85, len(ordered)))
for patch, color in zip(bp['boxes'], palette):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_xticklabels(ordered, rotation=25, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('5-Fold Cross-Validation Accuracy Distribution', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(min([s.min() for s in cv_scores.values()]) - 0.01, 1.005)
plt.tight_layout()
save_fig(fig, 'fig06_cv_boxplot')
plt.show()

### Figure 7 — Performance Metrics Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(results_df))
w = 0.2
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-score']
colors_m = ['#3498DB', '#2ECC71', '#F39C12', '#E74C3C']

for i, m in enumerate(metrics):
    offset = (i - 1.5) * w
    bars = ax.bar(x + offset, results_df[m], w, label=m,
                  color=colors_m[i], edgecolor='black', linewidth=0.4)

ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=20, ha='right')
ax.set_ylabel('Score')
ax.set_title('Performance Comparison of ML Algorithms', fontweight='bold')
ax.legend(loc='lower left', ncol=4, frameon=True)
ax.set_ylim(min(results_df[metrics].min()) - 0.05, 1.02)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
save_fig(fig, 'fig07_performance_bars')
plt.show()

### Figure 8 — Training Time vs Accuracy Trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
palette = plt.cm.tab10(np.linspace(0, 1, len(results_df)))
for i, row in results_df.iterrows():
    ax.scatter(row['Train_time'], row['Accuracy'],
               s=180, color=palette[i], edgecolor='black', linewidth=1, zorder=3,
               label=row['Model'])
    ax.annotate(row['Model'], (row['Train_time'], row['Accuracy']),
                xytext=(7, 5), textcoords='offset points', fontsize=8)
ax.set_xscale('log')
ax.set_xlabel('Training Time (seconds, log scale)')
ax.set_ylabel('Test Accuracy')
ax.set_title('Training Time vs Accuracy Trade-off', fontweight='bold')
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
save_fig(fig, 'fig08_time_vs_accuracy')
plt.show()

## 5. Best Model Analysis

### Figure 9 — Confusion Matrix

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model, needs_scaling = trained[best_name]
Xte = X_test_scaled if needs_scaling else X_test.values
y_pred_best = best_model.predict(Xte)

print(f'Best model: {best_name} (accuracy = {results_df.iloc[0]["Accuracy"]:.4f})')

cm = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label':'Count'}, linewidths=0.3, linecolor='lightgray',
            annot_kws={'fontsize': 8}, ax=ax)
ax.set_xlabel('Predicted Class', fontweight='bold')
ax.set_ylabel('True Class', fontweight='bold')
ax.set_title(f'Confusion Matrix — {best_name}', fontweight='bold', pad=10)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
save_fig(fig, 'fig09_confusion_matrix')
plt.show()

print('\n=== Classification Report ===\n')
print(classification_report(y_test, y_pred_best, target_names=class_names, digits=4))

### Figure 10 — Feature Importance

In [ ]:
# Use Random Forest for feature importance (interpretable, even if best is XGB)
fi_model = trained.get('Random Forest', (None, False))[0]
if fi_model is None or not hasattr(fi_model, 'feature_importances_'):
    fi_model = trained['XGBoost'][0]

importances = pd.Series(fi_model.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
colors = plt.cm.plasma(np.linspace(0.2, 0.85, len(importances)))
bars = ax.barh(importances.index, importances.values, color=colors, edgecolor='black', linewidth=0.5)
for bar, v in zip(bars, importances.values):
    ax.text(v + 0.005, bar.get_y() + bar.get_height()/2, f'{v:.3f}',
            va='center', fontsize=9)
ax.set_xlabel('Importance Score')
ax.set_title(f'Feature Importance (Random Forest)', fontweight='bold')
ax.set_xlim(0, importances.max() * 1.15)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
save_fig(fig, 'fig10_feature_importance')
plt.show()

### Figure 11 — ROC Curves (One-vs-Rest, multi-class)

In [ ]:
# Binarize labels for one-vs-rest ROC
n_classes = len(class_names)
y_test_bin = label_binarize(y_test, classes=np.arange(n_classes))
y_score = best_model.predict_proba(Xte)

# Compute ROC curve and AUC for each class
fpr, tpr, roc_auc = {}, {}, {}
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Macro-average
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes
fpr['macro'], tpr['macro'] = all_fpr, mean_tpr
roc_auc['macro'] = auc(all_fpr, mean_tpr)

fig, ax = plt.subplots(figsize=(8.5, 6.5))
palette = sns.color_palette('tab20', n_colors=n_classes)
for i in range(n_classes):
    ax.plot(fpr[i], tpr[i], color=palette[i], lw=1, alpha=0.7,
            label=f'{class_names[i]} (AUC={roc_auc[i]:.2f})')
ax.plot(fpr['macro'], tpr['macro'], color='black', lw=2.5, linestyle='--',
        label=f'macro-avg (AUC={roc_auc["macro"]:.3f})')
ax.plot([0, 1], [0, 1], 'k:', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curves (One-vs-Rest) — {best_name}', fontweight='bold')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=7, ncol=1)
ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.02)
ax.grid(alpha=0.3)
plt.tight_layout()
save_fig(fig, 'fig11_roc_curves')
plt.show()
print(f'\nMacro-average AUC: {roc_auc["macro"]:.4f}')

### Figure 12 — Learning Curve for Best Model

In [ ]:
# Learning curve: how accuracy scales with training set size
print('Computing learning curve (may take ~20-30s)...')
Xfull = X_train_scaled if needs_scaling else X_train.values
train_sizes_abs = np.linspace(0.1, 1.0, 8)
train_sizes, train_scores, val_scores = learning_curve(
    best_model, Xfull, y_train,
    train_sizes=train_sizes_abs, cv=5, scoring='accuracy',
    n_jobs=-1, random_state=42
)

train_mean, train_std = train_scores.mean(axis=1), train_scores.std(axis=1)
val_mean, val_std = val_scores.mean(axis=1), val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_mean, 'o-', color='#3498DB', lw=2, label='Training accuracy')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='#3498DB')
ax.plot(train_sizes, val_mean, 's-', color='#E74C3C', lw=2, label='Validation accuracy')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='#E74C3C')
ax.set_xlabel('Number of Training Samples')
ax.set_ylabel('Accuracy')
ax.set_title(f'Learning Curve — {best_name}', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
save_fig(fig, 'fig12_learning_curve')
plt.show()

## 6. Proposed Ensemble Approach

### Figure 13 — Ensemble Architecture

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xlim(0, 11); ax.set_ylim(0, 7); ax.axis('off')

# Input layer (centered)
draw_box(ax, (4.0, 5.7), 3, 0.8,
         'Input Features\n(N, P, K, Temperature, Humidity, pH, Rainfall)',
         '#AED6F1', fontsize=9, fontweight='bold')

# 4 base learners (evenly spaced)
base_y = 3.5
base_w = 2.1
base_h = 1.0
base_learners = [
    {'xy': (0.4, base_y), 'text': 'Random Forest\nClassifier',     'color': '#A9DFBF'},
    {'xy': (2.9, base_y), 'text': 'Naive Bayes\nClassifier',       'color': '#F8C471'},
    {'xy': (5.4, base_y), 'text': 'XGBoost\nClassifier',           'color': '#F9E79F'},
    {'xy': (7.9, base_y), 'text': 'Gradient Boosting\nClassifier', 'color': '#F5B7B1'},
]
for bl in base_learners:
    draw_box(ax, bl['xy'], base_w, base_h, bl['text'], bl['color'], fontsize=8)
    cx = bl['xy'][0] + base_w / 2
    draw_arrow(ax, (5.5, 5.7), (cx, base_y + base_h))

# Ensemble layer
draw_box(ax, (4.0, 1.5), 3, 1.0, 'Soft Voting\nEnsemble', '#D2B4DE',
         fontsize=10, fontweight='bold')
for bl in base_learners:
    sx = bl['xy'][0] + base_w / 2
    draw_arrow(ax, (sx, base_y), (5.5, 2.5))

# Output
draw_box(ax, (4.0, 0.1), 3, 0.8, 'Recommended Crop', '#FAD7A0',
         fontsize=10, fontweight='bold')
draw_arrow(ax, (5.5, 1.5), (5.5, 0.9))

# Layer labels
ax.text(0.05, 6.1, 'Input Layer',              fontsize=9, style='italic', color='gray')
ax.text(0.05, 4.0, 'Base Learners\n(Layer 1)', fontsize=9, style='italic', color='gray')
ax.text(0.05, 2.0, 'Ensemble Layer',           fontsize=9, style='italic', color='gray')
ax.text(0.05, 0.5, 'Output',                   fontsize=9, style='italic', color='gray')

plt.title('Proposed Ensemble Architecture', fontsize=11, fontweight='bold', pad=10)
plt.tight_layout()
save_fig(fig, 'fig13_ensemble_architecture')
plt.show()

### 6.1 Train the proposed ensemble and compare

In [ ]:
ensemble = VotingClassifier(
    estimators=[
        ('rf',  RandomForestClassifier(n_estimators=100, random_state=42)),
        ('nb',  GaussianNB()),
        ('xgb', XGBClassifier(eval_metric='mlogloss', random_state=42)),
        ('gb',  GradientBoostingClassifier(random_state=42)),
    ],
    voting='soft', n_jobs=-1
)
t0 = time.time()
ensemble.fit(X_train.values, y_train)
ens_train_time = time.time() - t0
y_ens = ensemble.predict(X_test.values)

ens_row = {
    'Model':      'Proposed Ensemble (RF+NB+XGB+GB, soft voting)',
    'Accuracy':   accuracy_score(y_test, y_ens),
    'Precision':  precision_score(y_test, y_ens, average='weighted'),
    'Recall':     recall_score(y_test, y_ens, average='weighted'),
    'F1-score':   f1_score(y_test, y_ens, average='weighted'),
    'Train_time': ens_train_time,
}
final_table = pd.concat([results_df, pd.DataFrame([ens_row])], ignore_index=True)\
                .sort_values('Accuracy', ascending=False).reset_index(drop=True)
print(final_table.round(4))
final_table.round(4).to_csv(f'{FIG_DIR}/table2_final_with_ensemble.csv', index=False)

In [ ]:
# =====================================================================
# REVISION (R1): Hardware profiling — model size + inference latency
# =====================================================================
import pickle

def model_size_kb(model):
    return len(pickle.dumps(model)) / 1024

def inference_latency_ms(model, X_sample, n_runs=1000):
    x1 = X_sample[:1]
    for _ in range(10):                 # warm up
        _ = model.predict(x1)
    t0 = time.perf_counter()
    for _ in range(n_runs):
        _ = model.predict(x1)
    return (time.perf_counter() - t0) / n_runs * 1000

print("=== HARDWARE PROFILE (Table 4 in paper) ===\n")
print(f"{'Model':22s} {'Size (KB)':>10s} {'Latency (ms/sample)':>20s}")
print("-" * 54)

# Individual models — unpack (model, needs_scaling) tuples
for name in ['Logistic Regression', 'Naive Bayes', 'XGBoost',
             'Random Forest', 'Gradient Boosting']:
    model, needs_scaling = trained[name]
    X_in = X_test_scaled if needs_scaling else X_test.values
    sz   = model_size_kb(model)
    lat  = inference_latency_ms(model, X_in)
    print(f"{name:22s} {sz:10.1f} {lat:20.3f}")

# Ensemble — trained on raw .values, so use raw test features
sz_ens  = model_size_kb(ensemble)
lat_ens = inference_latency_ms(ensemble, X_test.values)
print(f"{'Proposed Ensemble':22s} {sz_ens:10.1f} {lat_ens:20.3f}")

In [ ]:
# =====================================================================
# REVISION (R1): Bagging ensemble comparison across 10 seeds
# =====================================================================
from sklearn.ensemble import BaggingClassifier

print("=== BAGGING ROBUSTNESS (Table 3 row in paper) ===\n")
bag_acc, bag_f1 = [], []
for s in range(10):
    Xtr, Xte, ytr, yte = train_test_split(
        X, y_enc, test_size=0.2, random_state=s, stratify=y_enc)
    bag = BaggingClassifier(n_estimators=100, random_state=42)
    bag.fit(Xtr.values, ytr)
    yp = bag.predict(Xte.values)
    bag_acc.append(accuracy_score(yte, yp))
    bag_f1.append(f1_score(yte, yp, average='weighted'))

print(f"Bagging (homogeneous, 100 trees):")
print(f"  Accuracy: mean={np.mean(bag_acc):.4f}, std={np.std(bag_acc):.4f}")
print(f"  F1-score: mean={np.mean(bag_f1):.4f}, std={np.std(bag_f1):.4f}")
print()
var_reduction = (np.std(bag_acc) - 0.0021) / np.std(bag_acc) * 100
print(f"Ensemble reduces variance vs Bagging by: {var_reduction:.0f}%")

## 7. Download Everything (figures + tables)

Zips all figures (PNG + PDF) and CSV tables for easy download.

In [ ]:
zip_name = 'IEEE_paper_artifacts_1.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(os.listdir(FIG_DIR)):
        zf.write(f'{FIG_DIR}/{f}', arcname=f)
print(f'Created {zip_name} containing:')
for f in sorted(os.listdir(FIG_DIR)):
    sz = os.path.getsize(f'{FIG_DIR}/{f}') / 1024
    print(f'  {f:45s}  {sz:7.1f} KB')


In [ ]:
# =====================================================================
# Robustness experiment: stability across 10 random seeds
# Tests whether the proposed ensemble is more stable than individual
# top classifiers (RF, NB, XGB) when the train/test split varies.
# =====================================================================
import time
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

print("Running 10-seed robustness experiment (~6-8 minutes on Colab CPU)...\n")
seeds = list(range(10))
records = []
t_start = time.time()

for s in seeds:
    print(f"  seed={s} ...", end=" ", flush=True)
    t0 = time.time()

    # Re-split with this seed (same 80/20 stratified ratio)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y_enc, test_size=0.2, random_state=s, stratify=y_enc
    )

    # Individual classifiers (model random_state fixed; only split varies)
    rf  = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr.values, y_tr)
    nb  = GaussianNB().fit(X_tr.values, y_tr)
    xgb = XGBClassifier(eval_metric='mlogloss', random_state=42,
                        ).fit(X_tr.values, y_tr)

    # 4-model ensemble
    ens = VotingClassifier(
        estimators=[
            ('rf',  RandomForestClassifier(n_estimators=100, random_state=42)),
            ('nb',  GaussianNB()),
            ('xgb', XGBClassifier(eval_metric='mlogloss', random_state=42)),
            ('gb',  GradientBoostingClassifier(random_state=42)),
        ],
        voting='soft', n_jobs=-1
    ).fit(X_tr.values, y_tr)

    # Record accuracy and F1 for all four
    for name, mdl in [('Random Forest', rf), ('Naive Bayes', nb),
                      ('XGBoost', xgb), ('Proposed Ensemble', ens)]:
        yp = mdl.predict(X_te.values)
        records.append({
            'seed': s,
            'model': name,
            'accuracy': accuracy_score(y_te, yp),
            'f1':       f1_score(y_te, yp, average='weighted')
        })
    print(f"({time.time()-t0:.1f}s)")

print(f"\nTotal time: {(time.time()-t_start)/60:.1f} min")

# Summary table
rob_df = pd.DataFrame(records)
summary = rob_df.groupby('model').agg(
    acc_mean=('accuracy','mean'),
    acc_std =('accuracy','std'),
    acc_min =('accuracy','min'),
    acc_max =('accuracy','max'),
    f1_mean =('f1','mean'),
    f1_std  =('f1','std'),
).round(4)
order = ['Proposed Ensemble', 'Random Forest', 'Naive Bayes', 'XGBoost']
summary = summary.reindex(order)
print('\n=== TABLE III: Robustness across 10 random seeds ===\n')
print(summary)
summary.to_csv(f'{FIG_DIR}/table3_robustness.csv')

# Visualization: per-seed accuracy distribution
fig, ax = plt.subplots(figsize=(8, 5))
data = [rob_df[rob_df['model']==m]['accuracy'].values for m in order]
bp = ax.boxplot(data, labels=order, patch_artist=True, widths=0.6, showmeans=True,
                meanprops={'marker':'D', 'markerfacecolor':'red', 'markeredgecolor':'red'})
for patch, color in zip(bp['boxes'], ['#D2B4DE','#A9DFBF','#F8C471','#F9E79F']):
    patch.set_facecolor(color); patch.set_alpha(0.75)
ax.set_xticklabels(order, rotation=12, ha='right')
ax.set_ylabel('Test Accuracy')
ax.set_title('Accuracy Distribution Across 10 Random Seeds', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
save_fig(fig, 'fig05_robustness_seeds')
plt.show()

In [ ]:
# Also render Table III as PDF + PNG for visual archival
fig, ax = plt.subplots(figsize=(9.5, 2.2))
ax.axis('off')

col_labels = ['Model', 'Acc. Mean', 'Acc. Std', 'Acc. Min', 'Acc. Max', 'F1 Mean', 'F1 Std']
cell_text = []
for idx in summary.index:
    r = summary.loc[idx]
    cell_text.append([
        idx,
        f'{r["acc_mean"]:.4f}',
        f'{r["acc_std"]:.4f}',
        f'{r["acc_min"]:.4f}',
        f'{r["acc_max"]:.4f}',
        f'{r["f1_mean"]:.4f}',
        f'{r["f1_std"]:.4f}',
    ])

tbl = ax.table(cellText=cell_text, colLabels=col_labels,
               loc='center', cellLoc='center', colLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.6)

# Style header
for j in range(len(col_labels)):
    tbl[(0, j)].set_facecolor('#4472C4')
    tbl[(0, j)].set_text_props(weight='bold', color='white')

# Highlight the Proposed Ensemble row (row 1, since 0 is header)
for j in range(len(col_labels)):
    tbl[(1, j)].set_facecolor('#D2B4DE')
    tbl[(1, j)].set_text_props(weight='bold')

plt.title('TABLE III: Robustness Across 10 Random Seeds',
          fontweight='bold', pad=10)
plt.tight_layout()
save_fig(fig, 'table3_robustness')
plt.show()

## 8. Mapping Figures to Paper Sections (IEEE 8-page template)

| Paper section | What to include | Length |
|---------------|------------------|--------|
| **Abstract** | 150-200 words: motivation, dataset, methods compared, best result, conclusion | 1 paragraph |
| **I. Introduction** | Importance of precision agriculture in India; problem statement; contributions (3-4 bullets) | ~3/4 page |
| **II. Related Work** | 15-20 references, organized by approach (DL, classical ML, hybrid). Use citations from your survey (Paper 1) | ~3/4 page |
| **III. Methodology** | **Fig 1** (flowchart) + **Figs 2-5** (dataset) + algorithm descriptions (one paragraph each, total 8 paragraphs) | ~2 pages |
| **IV. Results & Discussion** | **Table I** + **Figs 6-12**. Discuss: which model won, why, edge cases (confused crops), time-accuracy tradeoff | ~2 pages |
| **V. Proposed Approach** | **Fig 13** (ensemble architecture) + **Table II** (results with ensemble) | ~3/4 page |
| **VI. Conclusion & Future Work** | Recap best result; future: deep learning, real-time deployment, weather API integration, mobile app | ~1/3 page |
| **References** | 20-25 IEEE-format references | ~1 page |

### Citation tips
- Cite the dataset source: Ingle, A. "Crop Recommendation Dataset". Kaggle. 2020.
- Cite at least 5 papers using the same dataset (search IEEE Xplore for "crop recommendation machine learning")
- Cite original papers for each algorithm: Breiman 2001 (RF), Chen & Guestrin 2016 (XGBoost), Cortes & Vapnik 1995 (SVM)

### Where to submit
Conferences (hybrid, Scopus-via-IEEE-Xplore, AI/agriculture-friendly tracks):
- AIC 2026 (Jabalpur, Aug 29-30) — submission deadlines around July
- ICCMRAI 2026 (Pune, Sep 11-12) — submission deadlines around July-Aug
- ICST 2026 (IIT Patna) — verify dates on website
- ICDSA 2026 (VIT Mauritius) — Springer LNNS proceedings

Verify each on the conferences.ieee.org listing and the IEEE Xplore proceedings of their last edition before paying registration.